In [6]:
# -*- coding: utf-8 -*-

from pathlib import Path
import shutil
import random
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

NON_CANCER_FOLDER = Path(
    r"\\?\D:\Downloads\Image Preprocessing Project\Mammogram Mastery A Robust Dataset for Breast Cancer Detection and Medical Education (1)\Mammogram Mastery A Robust Dataset for Breast Cancer Detection and Medical Education\Breast Cancer Dataset\Original Dataset\Non-Cancer"
)

CANCER_FOLDER = Path(
    r"\\?\D:\Downloads\Image Preprocessing Project\Mammogram Mastery A Robust Dataset for Breast Cancer Detection and Medical Education (1)\Mammogram Mastery A Robust Dataset for Breast Cancer Detection and Medical Education\Breast Cancer Dataset\Original Dataset\Cancer"
)

OUTPUT_ROOT = Path(
    r"D:\Downloads\Image Preprocessing Project\breast_cancer_split_original"
)

CLASS_FOLDERS = {
    "Non-Cancer": NON_CANCER_FOLDER,
    "Cancer": CANCER_FOLDER,
}

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def list_images(folder):
    return [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]


def copy_files(files, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for src in files:
        shutil.copy2(src, destination / src.name)


def split_files(files):
    train_files, temp_files = train_test_split(
        files,
        train_size=TRAIN_RATIO,
        random_state=SEED,
        shuffle=True
    )

    val_files, test_files = train_test_split(
        temp_files,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=SEED,
        shuffle=True
    )

    return train_files, val_files, test_files


def main():
    split_data = {
        "train": {},
        "val": {},
        "test": {}
    }

    for class_name, class_folder in CLASS_FOLDERS.items():
        print(f"\nChecking {class_name}:")
        print(class_folder)
        print("Exists:", class_folder.exists())
        print("Is dir:", class_folder.is_dir())

        if not class_folder.exists():
            raise FileNotFoundError(f"Folder not found: {class_folder}")

        files = list_images(class_folder)

        print("Images found:", len(files))

        if len(files) == 0:
            print("\nFiles inside this folder:")
            for item in class_folder.iterdir():
                print("DIR " if item.is_dir() else "FILE", repr(item.name))

            raise ValueError(f"No valid images found in: {class_folder}")

        train_files, val_files, test_files = split_files(files)

        split_data["train"][class_name] = train_files
        split_data["val"][class_name] = val_files
        split_data["test"][class_name] = test_files

    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)

    for split_name in ["train", "val", "test"]:
        for class_name in CLASS_FOLDERS:
            destination = OUTPUT_ROOT / split_name / class_name
            copy_files(split_data[split_name][class_name], destination)

    print("\nSplit finished successfully.\n")

    for split_name in ["train", "val", "test"]:
        print(split_name.upper())
        for class_name in CLASS_FOLDERS:
            count = len(split_data[split_name][class_name])
            print(f"  {class_name}: {count}")
        print()


if __name__ == "__main__":
    main()


Checking Non-Cancer:
\\?\D:\Downloads\Image Preprocessing Project\Mammogram Mastery A Robust Dataset for Breast Cancer Detection and Medical Education (1)\Mammogram Mastery A Robust Dataset for Breast Cancer Detection and Medical Education\Breast Cancer Dataset\Original Dataset\Non-Cancer
Exists: True
Is dir: True
Images found: 620

Checking Cancer:
\\?\D:\Downloads\Image Preprocessing Project\Mammogram Mastery A Robust Dataset for Breast Cancer Detection and Medical Education (1)\Mammogram Mastery A Robust Dataset for Breast Cancer Detection and Medical Education\Breast Cancer Dataset\Original Dataset\Cancer
Exists: True
Is dir: True
Images found: 125

Split finished successfully.

TRAIN
  Non-Cancer: 434
  Cancer: 87

VAL
  Non-Cancer: 93
  Cancer: 19

TEST
  Non-Cancer: 93
  Cancer: 19

